In [2]:
import json
import random
from collections import defaultdict

# 1. Load the massive raw dataset
input_file = "extracted_chart_specs_balanced.json"
output_file = "training_data_balanced.json"

print(f"Loading {input_file}...")
with open(input_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 2. Group the specs by Chart Type and Complexity
grouped_data = defaultdict(lambda: defaultdict(list))

for spec in raw_data:
    c_type = spec.get("chart_type")
    complexity = spec.get("complexity")
    
    if c_type and complexity:
        grouped_data[c_type][complexity].append(spec)

# 3. Define the precision sampling targets 
# (Prioritizing "High" complexity for dense mathematical arrays)
sampling_targets = {
    "Box Plot": {"High": 9, "Medium": 72, "Low": 52},
    "Area Chart": {"High": 91, "Medium": 157, "Low": 97},
    "Histogram": {"High": 0, "Medium": 183, "Low": 220},
    "Line Chart": {"High": 396, "Medium": 77, "Low": 77},
    "Scatter Plot": {"High": 258, "Medium": 146, "Low": 146},
    "Bar Chart": {"High": 359, "Medium": 46, "Low": 45},
    "Pie Chart": {"High": 0, "Medium": 0, "Low": 150} 
}

balanced_dataset = []

# Lock the random seed so your thesis data split is 100% reproducible
random.seed(42) 

print("\nExecuting Stratified Downsampling:")
print("-" * 40)

# 4. Execute the sampling
for c_type, complexity_targets in sampling_targets.items():
    for comp_level, target_count in complexity_targets.items():
        available_specs = grouped_data[c_type][comp_level]
        
        if target_count > 0 and available_specs:
            # Safely take the target amount, or all available if target > available
            actual_take = min(target_count, len(available_specs))
            sampled = random.sample(available_specs, actual_take)
            balanced_dataset.extend(sampled)
            
            print(f"{c_type:<15} ({comp_level:<6}): Kept {actual_take:<4} / {len(available_specs):<4}")

# 5. Shuffle the dataset
# Prevents the VLM from learning in monolithic "blocks" of chart types
random.shuffle(balanced_dataset)

print("-" * 40)
print(f"✅ Original Dataset Size: {len(raw_data)} samples")
print(f"✅ Final Balanced Dataset Size: {len(balanced_dataset)} samples")

# 6. Save the final pristine dataset
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(balanced_dataset, f, indent=2)

print(f"\nSaved perfectly balanced dataset to: {output_file}")

Loading extracted_chart_specs_balanced.json...

Executing Stratified Downsampling:
----------------------------------------
Box Plot        (High  ): Kept 5    / 5   
Box Plot        (Medium): Kept 62   / 62  
Box Plot        (Low   ): Kept 47   / 47  
Area Chart      (High  ): Kept 91   / 91  
Area Chart      (Medium): Kept 157  / 157 
Area Chart      (Low   ): Kept 97   / 97  
Histogram       (Medium): Kept 183  / 183 
Histogram       (Low   ): Kept 220  / 220 
Line Chart      (High  ): Kept 396  / 473 
Line Chart      (Medium): Kept 77   / 119 
Line Chart      (Low   ): Kept 77   / 83  
Scatter Plot    (High  ): Kept 185  / 185 
Scatter Plot    (Medium): Kept 114  / 114 
Scatter Plot    (Low   ): Kept 145  / 145 
Bar Chart       (High  ): Kept 359  / 359 
Bar Chart       (Medium): Kept 46   / 46  
Bar Chart       (Low   ): Kept 45   / 45  
Pie Chart       (Low   ): Kept 150  / 150 
----------------------------------------
✅ Original Dataset Size: 2581 samples
✅ Final Balanced Datase

In [3]:
# Check the distribution of balanced dataset according to chart type and complexity
distribution_check = defaultdict(lambda: defaultdict(int))
for spec in balanced_dataset:
    c_type = spec.get("chart_type")
    complexity = spec.get("complexity")
    if c_type and complexity:
        distribution_check[c_type][complexity] += 1
print("\nFinal Distribution in Balanced Dataset:")
for c_type, complexity_counts in distribution_check.items():
    print(f"{c_type}: {dict(complexity_counts)}")



Final Distribution in Balanced Dataset:
Pie Chart: {'Low': 150}
Histogram: {'Low': 220, 'Medium': 183}
Line Chart: {'Medium': 77, 'High': 396, 'Low': 77}
Scatter Plot: {'High': 185, 'Low': 145, 'Medium': 114}
Area Chart: {'High': 91, 'Medium': 157, 'Low': 97}
Bar Chart: {'Medium': 46, 'High': 359, 'Low': 45}
Box Plot: {'Medium': 62, 'Low': 47, 'High': 5}


In [ ]:
import os
import json
import shutil

# Define paths
input_json = "training_data_balanced.json"
output_json = "training_data_balanced.json"
new_image_folder = "balanced_images"

# Create the new destination folder
os.makedirs(new_image_folder, exist_ok=True)

print(f"Loading {input_json}...")
with open(input_json, 'r', encoding='utf-8') as f:
    balanced_dataset = json.load(f)

successful_copies = 0
missing_images = 0

print("Copying images and updating JSON paths...")

for spec in balanced_dataset:
    original_img_path = spec.get("image")
    
    if original_img_path and os.path.exists(original_img_path):
        # Extract just the filename (e.g., '92762.png')
        filename = os.path.basename(original_img_path)
        new_img_path = os.path.join(new_image_folder, filename)
        
        # Copy the file to the new folder (preserves metadata)
        shutil.copy2(original_img_path, new_img_path)
        
        # Update the JSON entry to point to the new folder location
        spec["image"] = new_img_path
        successful_copies += 1
    else:
        print(f"⚠️ Warning: Image not found at {original_img_path}")
        missing_images += 1

# Save the updated JSON
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(balanced_dataset, f, indent=2)

print("-" * 40)
print(f"✅ Successfully copied {successful_copies} images into '{new_image_folder}/'")
if missing_images > 0:
    print(f"⚠️ Could not find {missing_images} images.")
print(f"✅ Saved updated paths to '{output_json}'")

Loading training_data_balanced.json...
Copying images and updating JSON paths...
----------------------------------------
✅ Successfully copied 2581 images into 'balanced_images/'
✅ Saved updated paths to 'training_data_balanced_final.json'


In [3]:
#load and check the final dataset
import json
from collections import defaultdict

final_training_data = []
with open("training_data_balanced.json", 'r', encoding='utf-8') as f:
    final_training_data = json.load(f)

# Get the average length of ChartSpec in the final dataset
average_length = sum(len(json.dumps(item.get("ChartSpec", {}))) for item in final_training_data) / len(final_training_data)
print(f"\nAverage length of ChartSpec in the final dataset: {average_length:.2f} characters")

# Get the median length of ChartSpec in the final dataset
lengths = sorted(len(json.dumps(item.get("ChartSpec", {}))) for item in final_training_data)
median_length = lengths[len(lengths) // 2] if len(lengths) % 2 == 1 else (lengths[len(lengths) // 2 - 1] + lengths[len(lengths) // 2]) / 2
print(f"Median length of ChartSpec in the final dataset: {median_length:.2f} characters")

# Get the length distribution of ChartSpec in the final dataset (0-1000, 1000-2000, 2000-3000, etc.)
length_distribution = defaultdict(int)
for item in final_training_data:
    length = len(json.dumps(item.get("ChartSpec", {})))
    if length < 1000:
        length_distribution["0-1000"] += 1
    elif length < 2000:
        length_distribution["1000-2000"] += 1
    elif length < 3000:
        length_distribution["2000-3000"] += 1
    elif length < 4000:
        length_distribution["3000-4000"] += 1
    elif length < 5000:
        length_distribution["4000-5000"] += 1
    elif length < 6000:
        length_distribution["5000-6000"] += 1
    elif length < 7000:
        length_distribution["6000-7000"] += 1
    elif length < 10000:
        length_distribution["7000-10000"] += 1
    else:
        length_distribution["10000+"] += 1

print("\nLength distribution of ChartSpec in the final dataset:")
for interval, count in length_distribution.items():
    print(f"{interval}: {count}")

# Get chart type distribution per length interval
chart_type_length_distribution = defaultdict(lambda: defaultdict(int))
for item in final_training_data:
    chart_type = item.get("chart_type", "Unknown")
    length = len(json.dumps(item.get("ChartSpec", {})))
    
    if length < 1000:
        chart_type_length_distribution["0-1000"][chart_type] += 1
    elif length < 2000:
        chart_type_length_distribution["1000-2000"][chart_type] += 1
    elif length < 3000:
        chart_type_length_distribution["2000-3000"][chart_type] += 1
    elif length < 4000:
        chart_type_length_distribution["3000-4000"][chart_type] += 1
    elif length < 5000:
        chart_type_length_distribution["4000-5000"][chart_type] += 1
    elif length < 6000:
        chart_type_length_distribution["5000-6000"][chart_type] += 1
    elif length < 7000:
        chart_type_length_distribution["6000-7000"][chart_type] += 1
    elif length < 10000:
        chart_type_length_distribution["7000-10000"][chart_type] += 1
    else:
        chart_type_length_distribution["10000+"][chart_type] += 1
print("\nChart type distribution per length interval:")
for interval, chart_counts in chart_type_length_distribution.items():
    print(f"{interval}:")
    for chart_type, count in chart_counts.items():
        print(f"  {chart_type}: {count}")

# List of ids for charts with 7000+ characters in ChartSpec
ids_7000_plus = defaultdict(list)
just_ids = []
for item in final_training_data:
    length = len(json.dumps(item.get("ChartSpec", {})))
    if length >= 7000:
        ids_7000_plus[item.get("id")].append(item.get("chart_type") + f" ({length} chars)")
        just_ids.append(item.get("id"))
print(f"\nIDs of charts with 7000+ characters in ChartSpec: {ids_7000_plus}")
print(f"IDs of charts with 7000+ characters in ChartSpec (just IDs): {just_ids}")

# quartile distribution of ChartSpec lengths
lengths = sorted(len(json.dumps(item.get("ChartSpec", {}))) for item in final_training_data)
q1 = lengths[len(lengths) // 4]
q2 = lengths[len(lengths) // 2]
q3 = lengths[3 * len(lengths) // 4]
print(f"\nQuartiles of ChartSpec lengths in the final dataset:")
print(f"Q1 (25th percentile): {q1} characters")
print(f"Q2 (Median): {q2} characters")
print(f"Q3 (75th percentile): {q3} characters")




Average length of ChartSpec in the final dataset: 2137.90 characters
Median length of ChartSpec in the final dataset: 1989.00 characters

Length distribution of ChartSpec in the final dataset:
2000-3000: 523
0-1000: 732
1000-2000: 565
3000-4000: 554
4000-5000: 175
6000-7000: 4
5000-6000: 26
7000-10000: 2

Chart type distribution per length interval:
2000-3000:
  Bar Chart: 56
  Area Chart: 96
  Histogram: 110
  Line Chart: 115
  Scatter Plot: 138
  Box Plot: 8
0-1000:
  Box Plot: 72
  Scatter Plot: 209
  Pie Chart: 147
  Histogram: 49
  Line Chart: 47
  Area Chart: 139
  Bar Chart: 69
1000-2000:
  Box Plot: 53
  Histogram: 239
  Scatter Plot: 140
  Area Chart: 28
  Bar Chart: 58
  Line Chart: 44
  Pie Chart: 3
3000-4000:
  Line Chart: 235
  Scatter Plot: 56
  Area Chart: 59
  Bar Chart: 201
  Histogram: 3
4000-5000:
  Line Chart: 81
  Bar Chart: 66
  Area Chart: 21
  Scatter Plot: 5
  Histogram: 2
6000-7000:
  Area Chart: 2
  Scatter Plot: 1
  Line Chart: 1
5000-6000:
  Line Chart: 26

In [2]:
# remove the 6000+ character charts from the final dataset and save a new JSON
filtered_dataset = [item for item in final_training_data if len(json.dumps(item.get("ChartSpec", {}))) < 6000]
with open("training_data_balanced_final_filtered.json", 'w', encoding='utf-8') as f:
    json.dump(filtered_dataset, f, indent=2)
print(f"\n✅ Saved filtered dataset (removing 6000+ char charts) to 'training_data_balanced_final_filtered.json' with {len(filtered_dataset)} samples")


✅ Saved filtered dataset (removing 6000+ char charts) to 'training_data_balanced_final_filtered.json' with 2515 samples


In [6]:
# check the pixel dimensions of the chart images in balanced_images folder
from PIL import Image
import os
# image_folder = "balanced_images"
# dimensions = defaultdict(int)
# for filename in os.listdir(image_folder):
#     if filename.endswith(".png"):
#         img_path = os.path.join(image_folder, filename)
#         with Image.open(img_path) as img:
#             dimensions[img.size] += 1

# print("Image dimensions distribution:")
# for size, count in dimensions.items():
#     print(f"  {size}: {count}")

# downsize the images in balanced_images folder to 512x512 and save them in a new folder called balanced_images_512
from PIL import Image
input_folder = "balanced_images"
output_folder = "balanced_images_512"
os.makedirs(output_folder, exist_ok=True)
for filename in os.listdir(input_folder):
    if filename.endswith(".png"):
        img_path = os.path.join(input_folder, filename)
        with Image.open(img_path) as img:
            img_resized = img.resize((512, 512))
            img_resized.save(os.path.join(output_folder, filename))